# **Documenting ICE**

---


In [1]:
!pip install folium
!pip install panel==1.7.2
!pip install duckdb
!pip install urlparser

  Using cached click-7.1.2-py2.py3-none-any.whl.metadata (2.9 kB)
Using cached click-7.1.2-py2.py3-none-any.whl (82 kB)
  Attempting uninstall: click
    Found existing installation: click 8.3.1
    Uninstalling click-8.3.1:
      Successfully uninstalled click-8.3.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
atproto 0.0.65 requires click<9,>=8.1.3, but you have click 7.1.2 which is incompatible.
typer-slim 0.20.0 requires click>=8.0.0, but you have click 7.1.2 which is incompatible.
fiona 1.10.1 requires click~=8.0, but you have click 7.1.2 which is incompatible.
flask 3.1.2 requires click>=8.1.3, but you have click 7.1.2 which is incompatible.
dask 2025.12.0 requires click>=8.1, but you have click 7.1.2 which is incompatible.
google-adk 1.21.0 requires click<9.0.0,>=8.1.8, but you have click 7.1.2 which is incompatible.
typer 0.20.0 requires click>=8.0.0

In [2]:
# To ignore unimporant system warnings
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import geopandas as gpd
import numpy as np
import requests
import time
import matplotlib.pyplot as plt
from shapely.geometry import Point, Polygon
import duckdb
import folium

# This is a library for accessing and parsing data through URLs
from urllib.parse import urlencode
import urllib.request, json

# A magic functin that renders the figure in a notebook
%matplotlib inline

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Bluesky API & Video Collection

In [4]:
!pip install atproto requests
!pip install requests pandas python-dateutil

  Using cached click-8.3.1-py3-none-any.whl.metadata (2.6 kB)
Using cached click-8.3.1-py3-none-any.whl (108 kB)
  Attempting uninstall: click
    Found existing installation: click 7.1.2
    Uninstalling click-7.1.2:
      Successfully uninstalled click-7.1.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
urlparser 0.1.2 requires click<8.0,>=6.0, but you have click 8.3.1 which is incompatible.


In [5]:
!pip install -U spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 12.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [6]:
from atproto import Client
from getpass import getpass

HANDLE = "christiancarvajal.bsky.social"          # My Bluesky handle
APP_PASSWORD = "cgni-a2nc-aqz3-4rnb"     # App password I created through Bluesky

client = Client()

# Log in
client.login(HANDLE, APP_PASSWORD)

# Fetch a few posts from your home timeline
timeline = client.app.bsky.feed.get_timeline(params={'limit': 5})

for item in timeline.feed:
    post = item.post
    text = post.record.text
    author = post.author.handle
    print(f"@{author}: {text[:80]!r}")

@alexbenzer.com: 'backend eng? come build this app with us.\n\njobs.gem.com/bluesky/am9i...\n\nwe’re a'
@bsky.app: '📢 v1.113 is rolling out now!\n\nThis release focuses on fixing bugs and improving '
@bsky.app: 'Want to discover great posts beyond who you already follow? The For You feed, bu'
@bsky.app: '他のソーシャルメディアで、Bluesky で一緒に年明けを祝おうよ！と、お正月フィードを共有してくださればとても嬉しいです。\nご質問がありましたら、日本担当カン'


In [7]:
import requests
import sqlite3
from dateutil import parser as dateparser
from datetime import datetime, timedelta, timezone

SEARCH_QUERIES = [
    "ICE encounter",
    "ICE arrest",
    "ICE raid",
    "ICE agent",
    "ICE attack",
    "ICE kidnap",
    "Immigration officer",
    "Masked federal officer"
]
PER_PAGE = 100
LANG = "en"
MAX_PAGES = 25

VIDEO_HOSTS = (
    "youtube.com",
    "youtu.be",
    "tiktok.com",
    "twitter.com",
    "x.com",
    "vimeo.com",
    ".mp4",
    ".m3u8"
)

CUTOFF_DATE = datetime(2025, 1, 20, tzinfo=timezone.utc)

SEEN_URIS = set()

ACRONYMS = ["ICE"]  # always exact match

def text_matches_ice_phrases(text: str) -> bool:
    """
    Returns True if text contains:
      - Any of the acronyms (exact match, case-sensitive)
      - OR any of the SEARCH_QUERIES (lemmatized, plurals/gerunds handled)
    """
    if not text:
        return False

    # Check acronyms exactly (case-sensitive)
    for acronym in ACRONYMS:
        if acronym in text:
            return True

    # Lemmatized check for other phrases
    text_lower = text.lower()
    doc = nlp(text_lower)
    lemmas = [token.lemma_ for token in doc]

    for phrase in SEARCH_QUERIES:
        # Skip acronyms
        if phrase in ACRONYMS:
            continue

        phrase_doc = nlp(phrase.lower())
        phrase_lemmas = [token.lemma_ for token in phrase_doc]

        # If all phrase lemmas appear somewhere in the text lemmas
        if all(pl in lemmas for pl in phrase_lemmas):
            return True

    return False

In [8]:
def search_bsky_posts_auth(query, limit=100, cursor=None, lang="en"):
    params = {
        'q': query,
        'limit': limit,
        'lang': lang,
    }
    if cursor:
        params['cursor'] = cursor

    result = client.app.bsky.feed.search_posts(params=params)

    return result.posts, result.cursor

In [9]:
def has_video_embed(post):
    """
    Detects native Bluesky video OR external video links
    at either view-level or record-level.
    """
    embeds = []

    # view-level embed
    if post.embed:
        embeds.append(post.embed)

    # record-level embed
    if post.record and post.record.embed:
        embeds.append(post.record.embed)

    for e in embeds:
        t = getattr(e, "py_type", "") or ""

        # Native video
        if "embed.video" in t:
            return True

        # External link
        if "embed.external" in t:
            uri = getattr(getattr(e, "external", None), "uri", "") or ""
            if any(h in uri for h in VIDEO_HOSTS):
                return True

        # Record-with-media
        if "recordWithMedia" in t:
            media = getattr(e, "media", None)
            if media and "video" in getattr(media, "py_type", ""):
                return True

    return False

In [10]:
import spacy

nlp = spacy.load("en_core_web_sm")

GEO_LABELS = {"GPE", "LOC", "FAC"}

def extract_locations_spacy(text: str):
    """
    Returns a list of geographic location strings inferred from text.
    """
    if not text:
        return []

    doc = nlp(text)
    locations = []

    for ent in doc.ents:
        if ent.label_ in GEO_LABELS:
            loc = ent.text.strip()
            if len(loc) >= 3:
                locations.append(loc)

    # Deduplicate while preserving order
    seen = set()
    out = []
    for l in locations:
        if l.lower() not in seen:
            seen.add(l.lower())
            out.append(l)

    return out

## Geocoding

In [11]:
!pip install geopy

from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

In [12]:
geolocator = Nominatim(user_agent="bluesky_video_mapper")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

GEOCODE_CACHE = {}

def geocode_location(name):
    if name in GEOCODE_CACHE:
        return GEOCODE_CACHE[name]

    try:
        loc = geocode(name, addressdetails=True, country_codes="us")
        if not loc:
            result = None
        else:
            address = loc.raw.get("address", {})
            country_code = address.get("country_code", "").lower()

            # US-only filter
            if country_code != "us":
                result = None
            else:
                result = (loc.latitude, loc.longitude)

    except Exception:
        result = None

    GEOCODE_CACHE[name] = result
    return result


## Pagination

In [13]:
all_results = []
SEEN_URIS = set()

print("Starting fresh run.")
print("Cutoff date:", CUTOFF_DATE.isoformat())

for query in SEARCH_QUERIES:
    print(f"\nQuery: {query}")
    cursor = None

    for page in range(MAX_PAGES):
        page_results = []
        posts, cursor = search_bsky_posts_auth(
            query,
            limit=PER_PAGE,
            cursor=cursor,
            lang=LANG
        )

        if not posts:
            break

        for post in posts:
            try:
                created_at = dateparser.parse(post.record.created_at).astimezone(timezone.utc)
            except Exception:
                continue

            if created_at < CUTOFF_DATE:
                continue

            if not has_video_embed(post):
                continue

            text = post.record.text or ""

            if not text_matches_ice_phrases(text):
                continue

            locations = extract_locations_spacy(text)

            try:
                did = post.author.did
                rkey = post.uri.split("/")[-1]
                post_url = f"https://bsky.app/profile/{did}/post/{rkey}"
            except Exception:
                continue

            if post_url in SEEN_URIS:
                continue

            SEEN_URIS.add(post_url)

            result = {
                "post_url": post_url,
                "created_at": created_at.isoformat(),
                "query": query,
                "text": text,
                "locations_inferred": locations,
                "locations_coords": []
            }

            # Geocode immediately
            for loc in locations:
                coords = geocode_location(loc)
                if coords:
                    result["locations_coords"].append({
                        "name": loc,
                        "lat": coords[0],
                        "lon": coords[1]
                    })

            # DROP posts with no US locations
            if not result["locations_coords"]:
                continue

            page_results.append(result)
            all_results.append(result)

        print(f"Page {page+1} | "
        f"Videos from this page: {len(page_results)}"
        )

        if not cursor:
            break

print(f"Total posts collected: {len(all_results)}")

Starting fresh run.
Cutoff date: 2025-01-20T00:00:00+00:00

Query: ICE encounter
Page 1 | Videos from this page: 2
Page 2 | Videos from this page: 3
Page 3 | Videos from this page: 3
Page 4 | Videos from this page: 2
Page 5 | Videos from this page: 1
Page 6 | Videos from this page: 1
Page 7 | Videos from this page: 1
Page 8 | Videos from this page: 0
Page 9 | Videos from this page: 2
Page 10 | Videos from this page: 2
Page 11 | Videos from this page: 2
Page 12 | Videos from this page: 0
Page 13 | Videos from this page: 0
Page 14 | Videos from this page: 1
Page 15 | Videos from this page: 1
Page 16 | Videos from this page: 2


Page 17 | Videos from this page: 4
Page 18 | Videos from this page: 0
Page 19 | Videos from this page: 0


Page 20 | Videos from this page: 4
Page 21 | Videos from this page: 0
Page 22 | Videos from this page: 0
Page 23 | Videos from this page: 1
Page 24 | Videos from this page: 1
Page 25 | Videos from this page: 1

Query: ICE arrest
Page 1 | Videos from this page: 1
Page 2 | Videos from this page: 3
Page 3 | Videos from this page: 4
Page 4 | Videos from this page: 5
Page 5 | Videos from this page: 0


Page 6 | Videos from this page: 2
Page 7 | Videos from this page: 3
Page 8 | Videos from this page: 4
Page 9 | Videos from this page: 0
Page 10 | Videos from this page: 3
Page 11 | Videos from this page: 1
Page 12 | Videos from this page: 1
Page 13 | Videos from this page: 0
Page 14 | Videos from this page: 1
Page 15 | Videos from this page: 7
Page 16 | Videos from this page: 3
Page 17 | Videos from this page: 2


Page 18 | Videos from this page: 4


...

Page 13 | Videos from this page: 5
Page 14 | Videos from this page: 6
Page 15 | Videos from this page: 6
Page 16 | Videos from this page: 7
Page 17 | Videos from this page: 4
Page 18 | Videos from this page: 3
Page 19 | Videos from this page: 1
Page 20 | Videos from this page: 10
Page 21 | Videos from this page: 13
Page 22 | Videos from this page: 4
Page 23 | Videos from this page: 2
Page 24 | Videos from this page: 2
Page 25 | Videos from this page: 2

Query: ICE agent
Page 1 | Videos from this page: 4
Page 2 | Videos from this page: 2
Page 3 | Videos from this page: 2
Page 4 | Videos from this page: 2
Page 5 | Videos from this page: 1
Page 6 | Videos from this page: 2
Page 7 | Videos from this page: 2
Page 8 | Videos from this page: 5
Page 9 | Videos from this page: 2
Page 10 | Videos from this page: 3
Page 11 | Videos from this page: 6
Page 12 | Videos from this page: 2
Page 13 | Videos from this page: 4


Page 14 | Videos from this page: 3
Page 15 | Videos from this page: 1
Page 16 | Videos from this page: 1
Page 17 | Videos from this page: 1
Page 18 | Videos from this page: 2
Page 19 | Videos from this page: 1
Page 20 | Videos from this page: 5
Page 21 | Videos from this page: 1
Page 22 | Videos from this page: 2
Page 23 | Videos from this page: 1
Page 24 | Videos from this page: 1
Page 25 | Videos from this page: 1

Query: ICE attack
Page 1 | Videos from this page: 2
Page 2 | Videos from this page: 1
Page 3 | Videos from this page: 1
Page 4 | Videos from this page: 3
Page 5 | Videos from this page: 2
Page 6 | Videos from this page: 0
Page 7 | Videos from this page: 3
Page 8 | Videos from this page: 0
Page 9 | Videos from this page: 1
Page 10 | Videos from this page: 3
Page 11 | Videos from this page: 1
Page 12 | Videos from this page: 2
Page 13 | Videos from this page: 1
Page 14 | Videos from this page: 12
Page 15 | Videos from this page: 0
Page 16 | Videos from this page: 2
Page 17 |

Page 14 | Videos from this page: 2
Page 15 | Videos from this page: 1
Page 16 | Videos from this page: 0
Page 17 | Videos from this page: 3
Page 18 | Videos from this page: 1
Page 19 | Videos from this page: 2
Page 20 | Videos from this page: 0
Page 21 | Videos from this page: 1
Page 22 | Videos from this page: 3
Page 23 | Videos from this page: 4
Page 24 | Videos from this page: 1
Page 25 | Videos from this page: 2

Query: Masked federal officer


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py", line 534, in _make_request
    response = conn.getresponse()
               ^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connection.py", line 565, in getresponse
    httplib_response = super().getresponse()
                       ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/http/client.py", line 1430, in getresponse
    response.begin()
  File "/usr/lib/python3.12/http/client.py", line 331, in begin
    version, status, reason = self._read_status()
                              ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/http/client.py", line 292, in _read_status
    line = str(self.fp.readline(_MAXLINE + 1), "iso-8859-1")
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socket.py", line 720, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/ssl.py", line

Page 1 | Videos from this page: 1
Page 2 | Videos from this page: 1
Page 3 | Videos from this page: 0
Total posts collected: 442


In [14]:
print("\nPOSTS WITH GEOCODED LOCATIONS\n")

for r in all_results[:5]:
    print("Post:", r["post_url"])
    print("Date:", r["created_at"])
    print("Query:", r["query"])
    print("Locations inferred:", r["locations_inferred"])
    print("Coordinates:", r["locations_coords"])
    print("Text:", r["text"][:120])
    print("-" * 70)

print(f"\nTOTAL POSTS COLLECTED: {len(all_results)}")



POSTS WITH GEOCODED LOCATIONS

Post: https://bsky.app/profile/did:plc:3i4p2vn5ugt4h5w245affthf/post/3mc73xp46ck2p
Date: 2026-01-12T03:29:25.066000+00:00
Query: ICE encounter
Locations inferred: ['North Carolina', 'U.S.']
Coordinates: [{'name': 'North Carolina', 'lat': 35.6729639, 'lon': -79.0392919}, {'name': 'U.S.', 'lat': 39.7837304, 'lon': -100.445882}]
Text: Two men stopped by ICE in North Carolina were U.S. citizens. When agents realized they were recording the encounter, the
----------------------------------------------------------------------
Post: https://bsky.app/profile/did:plc:wcx7bny6hb4h5toulnuo4h4l/post/3mc57oj4jss2w
Date: 2026-01-11T09:30:32.286000+00:00
Query: ICE encounter
Locations inferred: ['the United States']
Coordinates: [{'name': 'the United States', 'lat': 47.8291375, 'lon': -122.5970742}]
Text: Scary 😱 encounter with ICE. DoorDash delivery driver takes refuge in their home. Homeowners demand real warrant. Local s
---------------------------------------------

# Folium Map

In [15]:
import folium
from folium.plugins import MarkerCluster

In [16]:
import folium
from folium.plugins import MarkerCluster

def make_folium_map_with_query_toggle(results):
    # Center map on mean lat/lon
    coords = [
        (loc["lat"], loc["lon"])
        for r in results
        for loc in r["locations_coords"]
    ]
    if not coords:
        raise ValueError("No geocoded locations to map.")

    mean_lat = sum(c[0] for c in coords) / len(coords)
    mean_lon = sum(c[1] for c in coords) / len(coords)

    m = folium.Map(
        location=[mean_lat, mean_lon],
        zoom_start=4,
        tiles="CartoDB positron",
        max_bounds=True
    )

    # Create a FeatureGroup for each query
    query_groups = {}
    for query in set(r["query"] for r in results):
        query_groups[query] = folium.FeatureGroup(name=query)
        marker_cluster = MarkerCluster().add_to(query_groups[query])

        for r in results:
            if r["query"] != query:
                continue

            try:
                dt_obj = datetime.fromisoformat(r["created_at"])
                created_str = dt_obj.strftime("%b %d, %Y %H:%M UTC")
            except Exception:
                created_str = r["created_at"]

            for loc in r["locations_coords"]:

                # Build preview HTML
                popup_html = f"""
                <div style="max-width:300px; font-family:Arial, sans-serif;">
                    <b>Location:</b> {loc['name']}<br>
                    <b>Date:</b> {created_str}<br>
                    <b>Query:</b> {r['query']}<br>
                    <b>Post preview:</b><br>
                    <div style="margin-top:5px; margin-bottom:5px;">{r['text']}</div>
                """

                # Add link to actual Bluesky post
                popup_html += f'<a href="{r["post_url"]}" target="_blank">View full post</a>'
                popup_html += "</div>"

                query_colors = {
                "ICE encounter": "orange",
                "ICE arrest": "orange",
                "ICE raid": "red",
                "ICE attack": "red",
                "ICE kidnap": "red",
                "ICE agent": "cadetblue",
                "Immigration officer": "cadetblue",
                "Masked federal officer": "cadetblue"
                }

                folium.Marker(
                    location=[loc["lat"], loc["lon"]],
                    popup=folium.Popup(popup_html, max_width=320),
                    tooltip=loc["name"],
                    icon=folium.Icon(   # From FontAwesome
                        icon="exclamation-circle",
                        prefix="fa",
                        color=query_colors.get(r["query"], "gray")
                        )
                ).add_to(marker_cluster)

        query_groups[query].add_to(m)

    # Add LayerControl to toggle queries
    folium.LayerControl(collapsed=True).add_to(m)

    legend_html = """
    <div style="
    position: fixed;
    bottom: 30px;
    right: 10px;
    z-index: 9999;
    background: white;
    padding: 10px 12px;
    border-radius: 6px;
    box-shadow: 0 0 8px rgba(0,0,0,0.15);
    font-size: 13px;
    line-height: 1.4;
    ">
    <b>Bluesky Post Type</b><br>
    <hr style="margin:4px 0 6px 0;">
    """

    for query, color in query_colors.items():
        icon = "exclamation-circle"
        legend_html += f"""
        <div style="margin-bottom:4px;">
            <i class="fa fa-{icon}" style="color:{color}; width:18px;"></i>
            {query}
        </div>
        """

    legend_html += "</div>"

    m.get_root().html.add_child(folium.Element(legend_html))

    return m

In [17]:
m = make_folium_map_with_query_toggle(all_results)
m

In [20]:
m.save('/content/drive/MyDrive/01_Fall 2025/Urban Data & Informatics_Intro/Final Project_Documenting ICE/Final Deliverables/bluesky_ice_map.html')

# ACS 5-Year Data Collection & Clean Up
 Started with NYC specific. Then attempted by States but ran into connection issues. Leaving the code in here.

In [ ]:
API_KEY = "709be85ec89c41b6ee2627e05e0697f325f0456a"
STATE_FIPS = '36'
NYC_COUNTY_FIPS = ['005', '047', '061', '081', '085']

# Define Census Variables
CENSUS_VARIABLES = {

    # Place of Birth by Nativity and Citizenship Status
    'B05002_001E': 'B01001_001E',  # Total population
    'B05002_002E': 'B05002_002E',  # Native total
    'B05002_013E': 'B05002_013E',  # Foreign born total

    # Foreign-born, Not a U.S. citizen
    'B05002_021E': 'B05002_021E',  # Not a U.S. citizen
    'B05002_022E': 'B05002_022E',  # Europe
    'B05002_023E': 'B05002_023E',  # Asia
    'B05002_024E': 'B05002_024E',  # Africa
    'B05002_025E': 'B05002_025E',  # Oceania
    'B05002_026E': 'B05002_026E',  # Latin America
    'B05002_027E': 'B05002_027E',	 # Northern America

    # Worker Population by Workplace Geography
    'B08604_001E': 'B08604_001E',

    }

VARIABLE_CODES = list(CENSUS_VARIABLES.keys())

In [ ]:
# Data Fetching Function
def fetch_census_data(api_key, state_fips, county_fips_list, variables_dict):
    all_data = []
    variable_string = ','.join(variables_dict.keys()) + ',NAME'

    print(f"\nFetching data for {len(variables_dict)} variables...")

    for county_fips in county_fips_list:
        print(f"Fetching data for County FIPS {county_fips}...")
        url = (
            f"https://api.census.gov/data/2023/acs/acs5?"
            f"get={variable_string}&"
            f"for=tract:*&"
            f"in=state:{state_fips}&"
            f"in=county:{county_fips}&"
            f"key={api_key}"
        )

        try:
            response = requests.get(url)
            response.raise_for_status()
            data = response.json()
            headers = data[0]
            rows = data[1:]
            df = pd.DataFrame(rows, columns=headers)
            all_data.append(df)
            time.sleep(0.5)

        except Exception as e:
            print(f"Error for county {county_fips}: {e}. Skipping.")

    if not all_data:
        return pd.DataFrame()

    final_df = pd.concat(all_data, ignore_index=True)
    final_df['GEOID'] = final_df['state'] + final_df['county'] + final_df['tract']

    final_columns = ['GEOID', 'NAME'] + VARIABLE_CODES
    final_df = final_df[final_columns].copy()

    return final_df

In [ ]:
census_tract_data = fetch_census_data(API_KEY, STATE_FIPS, NYC_COUNTY_FIPS, CENSUS_VARIABLES)

census_tract_data


Fetching data for 11 variables...
Fetching data for County FIPS 005...


KeyboardInterrupt: 

In [ ]:
new_column_names = {
    'B05002_001E': 'Total population',
    'B05002_002E': 'Native total',
    'B05002_013E': 'Foreign born total',
    'B05002_021E': 'Not a U.S. citizen',
    'B05002_022E': 'Europe - Not Citizen',
    'B05002_023E': 'Asia - Not Citizen',
    'B05002_024E': 'Africa - Not Citizen',
    'B05002_025E': 'Oceania - Not Citizen',
    'B05002_026E': 'Latin America - Not Citizen',
    'B05002_027E': 'Northern America - Not Citizen',
    'B08604_001E': 'Worker Population by Workplace Geography',
}

census_tract_data2 = census_tract_data.rename(columns=new_column_names)

In [ ]:
census_tract_data2['GEOID'] = pd.to_numeric(census_tract_data2['GEOID'], errors='coerce').astype('Int64')

for col in census_tract_data2.columns[2:]:
    census_tract_data2[col] = pd.to_numeric(census_tract_data2[col], errors='coerce')

print(census_tract_data2.dtypes)
census_tract_data2.head(2)

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/01_Fall 2025/Urban Data & Informatics_Intro/Final Project_Documenting ICE/nyc2020census_tract_nta_cdta_relationships.csv')
df.head(2)

In [ ]:
df = df[['GEOID', 'BoroName', 'NTAName']]

df['GEOID'] = pd.to_numeric(df['GEOID'], errors='coerce').astype('Int64')

df.head(2)

In [ ]:
merged_df = pd.merge(census_tract_data2, df, on='GEOID', how='inner')

aggregation_rules_merged_df = {
    'GEOID': 'count',                   # Count of census tracts per NTAName
    'Total population': 'sum',
    'Native total': 'sum',
    'Foreign born total': 'sum',
    'Not a U.S. citizen': 'sum',
    'Europe - Not Citizen': 'sum',
    'Asia - Not Citizen': 'sum',
    'Africa - Not Citizen': 'sum',
    'Oceania - Not Citizen': 'sum',
    'Latin America - Not Citizen': 'sum',
    'Northern America - Not Citizen': 'sum',
    'Worker Population by Workplace Geography': 'sum',
    'BoroName': 'first',                # Assuming all tracts in a NTAName have the same BoroName
}

merged_df_by_nta = merged_df.groupby('NTAName').agg(aggregation_rules_merged_df).reset_index()

# Rename GEOID column to 'Number of Tracts'
merged_df_by_nta.rename(columns={'GEOID': 'Number of Census Tracts'}, inplace=True)

# Calculate 'Percent Foreign Born' based on the summed totals
merged_df_by_nta['Percent Foreign Born'] = (
    merged_df_by_nta['Foreign born total'] / merged_df_by_nta['Total population']
) * 100

demographics = merged_df_by_nta.copy()

display(demographics.head())

In [ ]:
Med = demographics['Percent Foreign Born'].median()

display(demographics['Percent Foreign Born'].describe())

print(f"\nAnd the median is {Med}.")

In [ ]:
merged_df_by_nta.sort_values(by='Percent Foreign Born', ascending=False).head(6)

In [ ]:
qr = """
    SELECT BoroName, COUNT(*) AS Immigrant_Neighborhoods
    FROM demographics
    WHERE "Percent Foreign Born" > 34.697
    GROUP BY BoroName
    ORDER BY Immigrant_Neighborhoods DESC
    """
ImmigrantNeighborhoods = duckdb.query(qr).df()
ImmigrantNeighborhoods

## Attempt to get similar data for states.

In [ ]:
API_KEY = "709be85ec89c41b6ee2627e05e0697f325f0456a"
ALL_STATE_FIPS = [
    '01', '02', '04', '05', '06', '08', '09', '10', '11', '12', '13', '16', '17',
    '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30',
    '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '44',
    '45', '46', '47', '48', '49', '50', '51', '53', '54', '55', '56'
]

# Define Census Variables
CENSUS_VARIABLES = {

    # Place of Birth by Nativity and Citizenship Status
    'B05002_001E': 'B05002_001E',
    'B05002_002E': 'B05002_002E',
    'B05002_013E': 'B05002_013E',
    'B05002_021E': 'B05002_021E',

    }

VARIABLE_CODES = list(CENSUS_VARIABLES.keys())

In [ ]:
# Data Fetching Function
def fetch_census_data(api_key, state_fips_list, variables_dict):
    all_data = []
    variable_string = ','.join(variables_dict.keys()) + ',NAME'

    print(f"\nFetching data for {len(variables_dict)} variables...")

    for state_fips in state_fips_list:
        print(f"Fetching data for State FIPS {state_fips}...")
        url = (
            f"https://api.census.gov/data/2023/acs/acs5?"
            f"get={variable_string}&"
            f"for=state:{state_fips}&"
            f"key={api_key}"
        )

        try:
            response = requests.get(url)
            response.raise_for_status()
            data = response.json()
            headers = data[0]
            rows = data[1:]
            df = pd.DataFrame(rows, columns=headers)
            all_data.append(df)
            time.sleep(0.5)

        except Exception as e:
            print(f"Error for state {state_fips}: {e}. Skipping.")

    if not all_data:
        return pd.DataFrame()

    final_df = pd.concat(all_data, ignore_index=True)
    final_df['GEOID'] = final_df['state']

    final_columns = ['GEOID', 'NAME'] + list(variables_dict.keys())
    final_df = final_df[final_columns].copy()

    return final_df

In [ ]:
census_state_data = fetch_census_data(API_KEY, ALL_STATE_FIPS, CENSUS_VARIABLES)
census_state_data

# Kept getting an error connecting. The idea was going to attempt to have a layer of foreign-born population as a percent for each state.


Fetching data for 4 variables...
Fetching data for State FIPS 01...


KeyboardInterrupt: 